# Description:

- This dataset contains top rated movies from IMDb website, gathered based on different genres and are also in different languages. Movies from the early 1930s to current year titles are collected, cleaned and arranged in this dataset.

#### Columns Description
 - 'Movie_Title' : Consist of 5000+ Movie Titles (5000+ Unique Values)
 - 'Year' : Ranging from 1920s to 2022 (99 Unique Values)
 - 'Director' : Names the Director (2000+ Unique Values)
 - 'Actors' : Names the Actors (5000+ Unique & Multiple Values)
 - 'Rating' : Titles rated for 10 by 25k+ Voters (74+ Unique Values)
 - 'main_genre' : Main Genre of the Title (13+ Unique Values)
 - 'side_genre' : Side / Multiple Genre of the Movie (144+ Unique & Multiple Values)
 - 'Runtime(Mins)' : Total duration of the movie in Minutes (156+ Unique Values)
 - 'Censor' : Censorship of the Movie (25+ Unique Values)
 - 'Total_Gross' : Total Box-Office Collection of the Movie (3500+ Unique Values)

### Dataset url : https://www.kaggle.com/datasets/rakkesharv/imdb-5000-movies-multiple-genres-dataset

# Importing the libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import plotly.express as px
import cufflinks as cf

# Step 1: Descriptive Statistics

In [ ]:
from pathlib import Path
import os

dataset_path = Path(
    os.environ.get(
        "IMDB_DATASET_PATH",
        "data/IMDb_All_Genres_etf_clean1.csv",
    )
).expanduser()
if not dataset_path.is_file():
    raise FileNotFoundError(
        f"Dataset not found: {dataset_path}. Place the CSV at "
        "data/IMDb_All_Genres_etf_clean1.csv or set IMDB_DATASET_PATH."
    )
df = pd.read_csv(dataset_path)


In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
df.info()

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.isnull().sum()

In [ ]:
df.shape

In [ ]:
for col in df.columns:
    print(f"{col}: {df[col].nunique()} unique values")

In [ ]:
def check_data(df):
    print("1. Missing Values Analysis:")
    print("-" * 50)

    print("NaN Values:")
    print(df.isna().sum())
    print("\n")

    print("Empty String Values (''):")
    empty_values = (df == '').sum()
    print(empty_values[empty_values > 0])
    print("\n")

    print("'NULL' Values:")
    null_values = (df == 'NULL').sum()
    print(null_values[null_values > 0])
    print("\n")

    print("'NA' Values:")
    na_values = (df == 'NA').sum()
    print(na_values[na_values > 0])
    print("\n")

    print("2. Numeric Columns Analysis:")
    print("-" * 50)
    numeric_columns = df.select_dtypes(include=['int64', 'float64']).columns

    for col in numeric_columns:
        print(f"\nAnalysis of Column {col}:")
        print(f"Negative Values: {(df[col] < 0).sum()}")
        print(f"Zero Values: {(df[col] == 0).sum()}")
        print(f"Minimum Value: {df[col].min()}")
        print(f"Maximum Value: {df[col].max()}")

    print("\n3. Text Columns Analysis:")
    print("-" * 50)
    text_columns = df.select_dtypes(include=['object']).columns

    for col in text_columns:
        print(f"\nAnalysis of Column {col}:")
        print(f"Unique Values Count: {df[col].nunique()}")
        print(f"Empty String Values (only spaces): {df[col].str.isspace().sum() if hasattr(df[col], 'str') else 'N/A'}")
        print(f"Shortest Value (character length): {df[col].str.len().min() if hasattr(df[col], 'str') else 'N/A'}")
        print(f"Longest Value (character length): {df[col].str.len().max() if hasattr(df[col], 'str') else 'N/A'}")

check_data(df)

In [ ]:
short_titles = df[df['Movie_Title'].str.len() <= 1]['Movie_Title']
print("Very short titles:", short_titles)

In [ ]:
long_directors = df[df['Director'].str.len() > 100]['Director']
print("Directors' long names:", long_directors)

In [ ]:
df.describe(include='all')

In [ ]:
df['Director'] = df['Director'].apply(lambda x: x[:100] if len(x) > 100 else x)
df['Movie_Title'] = df['Movie_Title'].apply(lambda x: 'Unknown Title' if len(x) <= 2 else x)

In [ ]:
df['Total_Gross'] = df['Total_Gross'].astype(str)
df['Total_Gross'] = df['Total_Gross'].replace('Gross Unkown', np.nan)
df['Total_Gross'] = df['Total_Gross'].replace({r'\$': '', ',': '', 'M': ''}, regex=True)
df['Total_Gross'] = pd.to_numeric(df['Total_Gross'], errors='coerce')
df['Total_Gross'] = df['Total_Gross'] * 1_000_000
df['Total_Gross'] = df['Total_Gross'].fillna(df['Total_Gross'].mean())
df['Total_Gross'] = df['Total_Gross'].replace(0, df['Total_Gross'].mean())

In [ ]:
df['Movie_Title'] = df['Movie_Title'].str.strip()
df['Director'] = df['Director'].str.strip()
df['Actors'] = df['Actors'].str.strip()

# Data Visualization

In [ ]:
cf.go_offline()
cf.set_config_file(world_readable=True, theme='pearl')

In [ ]:
line_plot = px.line(df, x='Year', y='Total_Gross', title='Total Gross Over Years')
line_plot.show()

In [ ]:
yearly_gross = df.groupby('Year')['Total_Gross'].sum().reset_index()
line_plot = px.line(yearly_gross, x='Year', y='Total_Gross', title='Total Gross Over Years')
line_plot.show()

In [ ]:
area_plot = df[['Runtime(Mins)', 'Total_Gross']].iplot(kind='area', title='Runtime vs Total Gross', asFigure=True)
area_plot.show()

In [ ]:
runtime_gross = df.groupby('Runtime(Mins)')['Total_Gross'].mean().reset_index()
area_plot = px.area(
    runtime_gross,
    x='Runtime(Mins)',
    y='Total_Gross',
    title='Runtime vs Total Gross',
    labels={'Runtime(Mins)': 'Runtime (Minutes)', 'Total_Gross': 'Total Gross'}
)
area_plot.show()

In [ ]:
hist_plot = px.histogram(df, x='Rating', nbins=20, title='Distribution of Movie Ratings', color_discrete_sequence=['skyblue'])
hist_plot.show()

In [ ]:
movies_per_year = df.groupby('Year').size().reset_index(name='Count')
bar_plot = px.bar(movies_per_year, x='Year', y='Count', title='Number of Movies Released Per Year')
bar_plot.show()


In [ ]:
pie_chart = px.pie(df, names='main_genre', title='Main Genre Distribution')
pie_chart.show()

In [ ]:
box_plot = px.box(df, x='main_genre', y='Rating', title='Rating Distribution by Main Genre', color='main_genre')
box_plot.show()

In [ ]:
scatter_plot = px.scatter(df, x='Rating', y='Total_Gross', color='Censor', title='Rating vs Total Gross Earnings', size='Total_Gross')
scatter_plot.show()

In [ ]:
bubble_plot = px.scatter(df, x='Runtime(Mins)', y='Rating', size='Total_Gross', color='Censor', title='Runtime vs Rating')
bubble_plot.show()

In [ ]:
bar_chart = df.groupby('main_genre')['Total_Gross'].sum().sort_values(ascending=False).iplot(
    kind='bar',
    title='Total Gross by Main Genre',
    xTitle='Main Genre',
    yTitle='Total Gross (in millions)',
    asFigure=True
)
bar_chart.show()

In [ ]:
bar_chart = df['Censor'].value_counts().reset_index().iplot(
    kind='bar',
    x='index',        
    y='Censor',        
    title='Distribution of Movies by Censor Ratings',
    xTitle='Censor Ratings',
    yTitle='Number of Movies',
    color='blue',
    theme='pearl',     
    asFigure=True
)

bar_chart.show()

# detect and remove outliers

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x='Total_Gross')
plt.title('Box Plot to Detect Outliers for Total Gross')
plt.show()

In [ ]:
Q1 = df['Total_Gross'].quantile(0.25)
Q3 = df['Total_Gross'].quantile(0.75)
IQR = Q3 - Q1

In [ ]:
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

In [ ]:
df_cleaned = df[(df['Total_Gross'] >= lower_bound) & (df['Total_Gross'] <= upper_bound)]

In [ ]:
print(f'Before cleaning: {df.shape[0]} rows')
print(f'After cleaning: {df_cleaned.shape[0]} rows')

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df_cleaned, x='Total_Gross')
plt.title('Box Plot After Removing Outliers for Total Gross')
plt.show()

#  Data Preprocessing

In [ ]:
df.head(10)

In [ ]:
df_cleaned = df.drop(columns=['Actors', 'Director', 'side_genre', 'Movie_Title'])
df_cleaned.head()

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(df_cleaned['Total_Gross'], kde=True)
plt.title('Distribution of Total Gross')
plt.show()


In [ ]:
df_cleaned['Censor'] = df_cleaned['Censor'].astype('category')
df_cleaned['main_genre'] = df_cleaned['main_genre'].astype('category')

# Feature Engineering

In [ ]:
# Do not derive predictors from Total_Gross, the regression target; that leaks the answer.
df_cleaned['Runtime_x_main_genre'] = df_cleaned['Runtime(Mins)'] * df_cleaned['main_genre'].cat.codes

In [ ]:
df_cleaned['Years_since_release'] = 2024 - df_cleaned['Year']

In [ ]:
# Revenue-per-minute uses Total_Gross (the target), so it is intentionally not a predictor.


In [ ]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
df_cleaned['Censor'] = label_encoder.fit_transform(df_cleaned['Censor'])

In [ ]:
df_cleaned = pd.get_dummies(df_cleaned, columns=['main_genre'], drop_first=True)
df_cleaned.head()

# Correlation Matrix

In [ ]:
correlation_matrix = df_cleaned.corr()
plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm')
plt.show()

# Discover Features

In [ ]:
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
X = df_cleaned.drop(['Total_Gross'], axis=1)
y = df_cleaned['Total_Gross']
selector = SelectKBest(score_func=f_regression, k=5)
X_new = selector.fit_transform(X, y)
selected_features = X.columns[selector.get_support()]
print("Top 5 Features: ", selected_features)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:
model = RandomForestRegressor(random_state=42)
model.fit(X, y)

In [ ]:
feature_importances = model.feature_importances_

In [ ]:
important_features = X.columns[feature_importances > 0.05]
print("Top Features: ", important_features)

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

In [ ]:
df_pca = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
print(df_pca.head())

In [ ]:
from sklearn.manifold import TSNE

In [ ]:
tsne = TSNE(n_components=2)
X_tsne = tsne.fit_transform(X)

In [ ]:
df_tsne = pd.DataFrame(X_tsne, columns=['TSNE1', 'TSNE2'])
df_tsne.head()

# Scaling 

In [ ]:
from sklearn.model_selection import train_test_split


In [ ]:
X = df_cleaned[['Runtime(Mins)', 'Runtime_x_main_genre', 'main_genre_Animation']]
y = df_cleaned['Total_Gross']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Keep preprocessing inside training-only workflows to avoid using held-out rows.


# Data Modelling

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

In [ ]:
y_pred_lr = lr_model.predict(X_test)

In [ ]:
mae_lr = mean_absolute_error(y_test, y_pred_lr)
mse_lr = mean_squared_error(y_test, y_pred_lr)
rmse_lr = mean_squared_error(y_test, y_pred_lr, squared=False)
r2_lr = r2_score(y_test, y_pred_lr)

print("Linear Regression - MAE:", mae_lr)
print("Linear Regression - MSE:", mse_lr)
print("Linear Regression - RMSE:", rmse_lr)
print("Linear Regression - R² Score:", r2_lr)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)

In [ ]:
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

In [ ]:
mae_rf = mean_absolute_error(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)
rmse_rf = mean_squared_error(y_test, y_pred_rf, squared=False)  
r2_rf = r2_score(y_test, y_pred_rf)

print("\nRandom Forest Regressor:")
print("MAE:", mae_rf)
print("MSE:", mse_rf)
print("RMSE:", rmse_rf)
print("R² Score:", r2_rf)

In [ ]:
from sklearn.model_selection import cross_val_score
scores_rf = cross_val_score(rf_model, X_train, y_train, cv=5, scoring='r2')
print("Cross Validation R² Scores:", scores_rf)
print("Mean Cross Validation R² Score:", scores_rf.mean())

In [ ]:
from sklearn.tree import DecisionTreeRegressor

In [ ]:
dt_model = DecisionTreeRegressor(random_state=42)

In [ ]:
dt_model.fit(X_train, y_train)

In [ ]:
y_pred_dt = dt_model.predict(X_test)

In [ ]:
mae_dt = mean_absolute_error(y_test, y_pred_dt)
mse_dt = mean_squared_error(y_test, y_pred_dt)
rmse_dt = mean_squared_error(y_test, y_pred_dt, squared=False)
r2_dt = r2_score(y_test, y_pred_dt)

In [ ]:
scores_dt = cross_val_score(dt_model, X_train, y_train, cv=5, scoring='r2')

In [ ]:
print("MAE:", mae_dt)
print("MSE:", mse_dt)
print("RMSE:", rmse_dt)
print("R² Score:", r2_dt)
print("Cross Validation R² scores:", scores_dt)
print("Mean Cross Validation R² score:", scores_dt.mean())

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
param_grid_dt = {
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5]
}

In [ ]:
grid_search_dt = GridSearchCV(dt_model, param_grid_dt, cv=5, scoring='r2')
grid_search_dt.fit(X_train, y_train)

In [ ]:
print("Best Parameters for Decision Tree:", grid_search_dt.best_params_)
print("Best Score:", grid_search_dt.best_score_)

In [ ]:
from sklearn.svm import SVR

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
svr_model = SVR(kernel='rbf')

In [ ]:
svr_model.fit(X_train_scaled, y_train)

In [ ]:
y_pred_svr = svr_model.predict(X_test_scaled)

In [ ]:
mae_svr = mean_absolute_error(y_test, y_pred_svr)
mse_svr = mean_squared_error(y_test, y_pred_svr)
rmse_svr = mean_squared_error(y_test, y_pred_svr, squared=False)
r2_svr = r2_score(y_test, y_pred_svr)

In [ ]:
print("\nSupport Vector Regressor:")
print("MAE:", mae_svr)
print("MSE:", mse_svr)
print("RMSE:", rmse_svr)
print("R² Score:", r2_svr)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn import metrics

def calculate_scaled_metrics(y_test, predictions):

    y_test_reshaped = y_test.to_numpy().reshape(-1, 1)
    predictions_reshaped = predictions.reshape(-1, 1)

    scaler = StandardScaler()
    scaled_Target = scaler.fit_transform(y_test_reshaped)
    pre_scaled_values = scaler.transform(predictions_reshaped)

    mae = metrics.mean_absolute_error(scaled_Target, pre_scaled_values)
    mse = metrics.mean_squared_error(scaled_Target, pre_scaled_values)
    rmse = np.sqrt(mse)

    print('MAE (Scaled):', mae)
    print('MSE (Scaled):', mse)
    print('RMSE (Scaled):', rmse)

print("\nLinear Regression:")
calculate_scaled_metrics(y_test, y_pred_lr)

print("\nRandom Forest Regressor:")
calculate_scaled_metrics(y_test, y_pred_rf)

print("\nDecision Tree Regressor:")
calculate_scaled_metrics(y_test, y_pred_dt)

print("\nSupport Vector Regressor:")
calculate_scaled_metrics(y_test, y_pred_svr)


In [ ]:
from scipy.stats import boxcox
y_train_transformed, lambda_ = boxcox(y_train + 1)


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVR

In [ ]:
parameters = {
    'kernel': ['rbf', 'linear', 'poly'],
    'C': [0.1, 1, 10, 100, 1000],
    'gamma': [0.001, 0.01, 0.1, 1, 'scale', 'auto'],
    'degree': [2, 3, 4]
}

In [ ]:
svr = SVR()

In [ ]:
grid_search = GridSearchCV(svr, parameters, scoring='neg_mean_absolute_error', cv=5)
grid_search.fit(X_train_scaled, y_train)

In [ ]:
best_svr = grid_search.best_estimator_
print("best param:", grid_search.best_params_)


In [ ]:
y_pred = best_svr.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)
med_ae = median_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

In [ ]:
print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("Median AE:", med_ae)
print("R² Score:", r2)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
from sklearn.feature_selection import SelectKBest, f_regression

selector = SelectKBest(score_func=f_regression, k=5)
X_train_reduced = selector.fit_transform(X_train_scaled, y_train)
X_test_reduced = selector.transform(X_test_scaled)


In [ ]:
svr_linear = SVR(kernel='linear', C=10)
svr_linear.fit(X_train_scaled, y_train)
y_pred_linear = svr_linear.predict(X_test_scaled)


In [ ]:
svm_model_scaled = grid_search.best_estimator_

y_pred_svm_scaled = svm_model_scaled.predict(X_test_scaled)
mae_svm_scaled = mean_absolute_error(y_test, y_pred_svm_scaled)
mse_svm_scaled = mean_squared_error(y_test, y_pred_svm_scaled)
r2_svm_scaled = r2_score(y_test, y_pred_svm_scaled)

print("\nSVM with Scaling - MAE:", mae_svm_scaled)
print("SVM with Scaling - MSE:", mse_svm_scaled)
print("SVM with Scaling - R² Score:", r2_svm_scaled)

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=5)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred_best = best_svr.predict(X_test_scaled)
print("MAE:", mean_absolute_error(y_test, y_pred_best))
print("MSE:", mean_squared_error(y_test, y_pred_best))
print("RMSE:", mean_squared_error(y_test, y_pred_best, squared=False))
print("R² Score:", r2_score(y_test, y_pred_best))


In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_train_normalized = scaler.fit_transform(X_train)
X_test_normalized = scaler.transform(X_test)

In [ ]:
svr_model = SVR(kernel='rbf', C=1, gamma='scale')
svr_model.fit(X_train_normalized, y_train)
y_pred = svr_model.predict(X_test_normalized)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("R² Score:", r2_score(y_test, y_pred))

In [ ]:
from sklearn.decomposition import PCA


In [ ]:
pca = PCA(n_components=5)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

In [ ]:
svm_model_pca = SVR()
svm_model_pca.fit(X_train_pca, y_train)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
y_pred_best = best_svr.predict(X_test_scaled)
print("MAE:", mean_absolute_error(y_test, y_pred_best))
print("MSE:", mean_squared_error(y_test, y_pred_best))
print("RMSE:", mean_squared_error(y_test, y_pred_best, squared=False))
print("R² Score:", r2_score(y_test, y_pred_best))

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

plt.figure(figsize=(8, 8))
ax = plt.axes(projection='3d')
ax.scatter(X_train_pca[:, 0], X_train_pca[:, 1], y_train, c='blue', label='Train Data')
ax.set_title('3D Visualization of PCA Reduced Data')
plt.legend()
plt.show()


In [ ]:
def calculate_scaled_metrics(y_test, predictions):
    y_test_reshaped = y_test.to_numpy().reshape(-1, 1)
    predictions_reshaped = predictions.reshape(-1, 1)

    scaler = StandardScaler()
    scaled_Target = scaler.fit_transform(y_test_reshaped)
    pre_scaled_values = scaler.transform(predictions_reshaped)

    mae = metrics.mean_absolute_error(scaled_Target, pre_scaled_values)
    mse = metrics.mean_squared_error(scaled_Target, pre_scaled_values)
    rmse = np.sqrt(mse)

    return mae, mse, rmse

metrics_data = {
    "Model": ["Linear", "Random Forest", "Decision Tree", "Support Vector"],
    "MAE": [
        calculate_scaled_metrics(y_test, y_pred_lr)[0],
        calculate_scaled_metrics(y_test, y_pred_rf)[0],
        calculate_scaled_metrics(y_test, y_pred_dt)[0],
        calculate_scaled_metrics(y_test, y_pred_svr)[0]
    ],
    "MSE": [
        calculate_scaled_metrics(y_test, y_pred_lr)[1],
        calculate_scaled_metrics(y_test, y_pred_rf)[1],
        calculate_scaled_metrics(y_test, y_pred_dt)[1],
        calculate_scaled_metrics(y_test, y_pred_svr)[1],
    ],
    "RMSE": [
        calculate_scaled_metrics(y_test, y_pred_lr)[2],
        calculate_scaled_metrics(y_test, y_pred_rf)[2],
        calculate_scaled_metrics(y_test, y_pred_dt)[2],
        calculate_scaled_metrics(y_test, y_pred_svr)[2],
    ],
}

df_metrics = pd.DataFrame(metrics_data)
fig, ax = plt.subplots(2, 2, figsize=(14, 10))

ax[0, 0].bar(df_metrics["Model"], df_metrics["MAE"], color='skyblue')
ax[0, 0].set_title("MAE")
ax[0, 0].set_xlabel("Model")

ax[0, 1].bar(df_metrics["Model"], df_metrics["MSE"], color='salmon')
ax[0, 1].set_title("MSE")
ax[0, 1].set_xlabel("Model")

ax[1, 0].bar(df_metrics["Model"], df_metrics["RMSE"], color='lightgreen')
ax[1, 0].set_title("RMSE")
ax[1, 0].set_xlabel("Model")

r2_scores = [
    r2_score(y_test, y_pred_lr),
    r2_score(y_test, y_pred_rf),
    r2_score(y_test, y_pred_dt),
    r2_score(y_test, y_pred_svr)
]
ax[1, 1].bar(df_metrics["Model"], r2_scores, color='orange')
ax[1, 1].set_title("R² Score")
ax[1, 1].set_xlabel("Model")

plt.subplots_adjust(hspace=0.3, wspace=0.3) 
plt.tight_layout()
plt.show()


# Check if normalization affects accuracy

In [ ]:
lr_model_no_norm = LinearRegression()
lr_model_no_norm.fit(X_train, y_train)
y_pred_no_norm = lr_model_no_norm.predict(X_test)
r2_no_norm = r2_score(y_test, y_pred_no_norm)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_model_norm = LinearRegression()
lr_model_norm.fit(X_train_scaled, y_train)
y_pred_norm = lr_model_norm.predict(X_test_scaled)
r2_norm = r2_score(y_test, y_pred_norm)

print("R² without normalization:", r2_no_norm)
print("R² with normalization:", r2_norm)


# Check if dimensionality reduction affects accuracy

In [ ]:
lr_model_no_pca = LinearRegression()
lr_model_no_pca.fit(X_train, y_train)
y_pred_no_pca = lr_model_no_pca.predict(X_test)
r2_no_pca = r2_score(y_test, y_pred_no_pca)

pca = PCA(n_components=0.95)  
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

lr_model_pca = LinearRegression()
lr_model_pca.fit(X_train_pca, y_train)
y_pred_pca = lr_model_pca.predict(X_test_pca)
r2_pca = r2_score(y_test, y_pred_pca)

print("R² without dimensionality reduction:", r2_no_pca)
print("R² with dimensionality reduction (PCA):", r2_pca)

In [ ]:
columns_to_drop = ['Censor'] + [col for col in df.columns if col.startswith('main_genre_')]
features = df.drop(columns=columns_to_drop, errors='ignore')

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

features_processed = features.copy()

columns_to_drop = ['Movie_Title', 'Director', 'Actors']
features_processed = features_processed.drop(columns=columns_to_drop, axis=1)
label_encoders = {}
for col in ['main_genre', 'side_genre']:
    le = LabelEncoder()
    features_processed[col] = le.fit_transform(features_processed[col])
    label_encoders[col] = le
print(features_processed.dtypes)
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features_processed)


In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numeric_columns = features.select_dtypes(include=['number'])

In [ ]:
scaler = StandardScaler()
scaled_features = scaler.fit_transform(numeric_columns)
import pandas as pd
scaled_features_df = pd.DataFrame(scaled_features, columns=numeric_columns.columns)

scaled_features_df.head()

In [ ]:
from sklearn.cluster import KMeans

n_classes = 3
kmeans = KMeans(n_clusters=n_classes, random_state=42)
clusters = kmeans.fit_predict(scaled_features)


In [ ]:
pca = PCA(n_components=2)
reduced_features = pca.fit_transform(scaled_features)

In [ ]:
plt.scatter(reduced_features[:, 0], reduced_features[:, 1], c=clusters, cmap='viridis')
plt.title('Cluster Visualization')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.colorbar(label='Cluster Label')
plt.show()

In [ ]:
plt.scatter(reduced_features[:, 0], reduced_features[:, 1], c=df['Rating'], cmap='viridis')
plt.title('Movies Visualization based on Rating')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.colorbar(label='Rating')
plt.show()

In [ ]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()

df['main_genre_encoded'] = label_encoder.fit_transform(df['main_genre'])

plt.scatter(reduced_features[:, 0], reduced_features[:, 1], c=df['main_genre_encoded'], cmap='viridis')
plt.title('Movies Visualization based on Main Genre')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.colorbar(label='Main Genre')
plt.show()

In [ ]:
from sklearn.metrics import adjusted_rand_score
ari = adjusted_rand_score(df['main_genre_encoded'], clusters)
print("Adjusted Rand Index:", ari)

In [ ]:
from sklearn.metrics import silhouette_score
silhouette = silhouette_score(scaled_features, clusters)
print("Silhouette Score:", silhouette)


### I hope you like it